In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
DATA_PATH = "hydrosense_phase2_labeled.csv"
MODEL_DIR = "trained_models"
RANDOM_STATE = 42
TEST_SIZE = 0.2

In [3]:
def load_and_clean_data(path):
    df = pd.read_csv(path)

    # remove weird extra header-like row
    df = df[df["timestamp"].notna()].copy()

    # remove unwanted text/footer rows if accidentally present
    df = df[df["zone_new"].notna()].copy()
    df = df[df["severity_new"].notna()].copy()

    # clean whitespace
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()

    return df

In [4]:
def prepare_features(df):
    # Use calibrated values for model training
    feature_cols_num = [
        "flow1_cal",
        "flow2_cal",
        "flow3_cal",
        "pressure",
        "diff12_cal",
        "diff23_cal",
        "ratio12_cal",
        "ratio23_cal",
    ]

    feature_cols_cat = [
        "operating_state",
    ]

    # keep only needed columns
    keep_cols = feature_cols_num + feature_cols_cat + ["zone_new", "severity_new"]
    df = df[keep_cols].copy()

    # convert numeric safely
    for col in feature_cols_num:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # some ratios may be inf or nan because of divide-by-zero cases
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # optional helper feature
    df["total_flow_cal"] = df["flow1_cal"] + df["flow2_cal"] + df["flow3_cal"]
    feature_cols_num.append("total_flow_cal")

    return df, feature_cols_num, feature_cols_cat

In [5]:
def train_model(df, target_col, num_cols, cat_cols, model_name):
    X = df[num_cols + cat_cols].copy()
    y = df[target_col].copy()

    # encode labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_encoded,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_encoded
    )

    # preprocessing
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ]
    )

    # model
    classifier = RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)

    # predictions
    y_pred = pipeline.predict(X_test)

    print(f"\n{'=' * 60}")
    print(f"{model_name.upper()} RESULTS")
    print(f"{'=' * 60}")
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    ))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return pipeline, label_encoder

In [6]:
os.makedirs(MODEL_DIR, exist_ok=True)

df = load_and_clean_data(DATA_PATH)
df, num_cols, cat_cols = prepare_features(df)

print("Dataset shape after cleaning:", df.shape)
print("\nZone distribution:")
print(df["zone_new"].value_counts())
print("\nSeverity distribution:")
print(df["severity_new"].value_counts())

# train zone model
zone_model, zone_encoder = train_model(
        df=df,
        target_col="zone_new",
        num_cols=num_cols,
        cat_cols=cat_cols,
        model_name="zone_model"
    )

# train severity model
severity_model, severity_encoder = train_model(
        df=df,
        target_col="severity_new",
        num_cols=num_cols,
        cat_cols=cat_cols,
        model_name="severity_model"
    )



Dataset shape after cleaning: (1134, 12)

Zone distribution:
LEAK_2_3    529
NORMAL      359
LEAK_1_2    246
Name: zone_new, dtype: int64

Severity distribution:
MEDIUM    465
LOW       359
HIGH      310
Name: severity_new, dtype: int64

ZONE_MODEL RESULTS
Accuracy: 0.9912

Classification Report:
              precision    recall  f1-score   support

    LEAK_1_2       0.98      0.98      0.98        49
    LEAK_2_3       0.99      0.99      0.99       106
      NORMAL       1.00      1.00      1.00        72

    accuracy                           0.99       227
   macro avg       0.99      0.99      0.99       227
weighted avg       0.99      0.99      0.99       227

Confusion Matrix:
[[ 48   1   0]
 [  1 105   0]
 [  0   0  72]]

SEVERITY_MODEL RESULTS
Accuracy: 0.9956

Classification Report:
              precision    recall  f1-score   support

        HIGH       1.00      1.00      1.00        62
         LOW       0.99      1.00      0.99        72
      MEDIUM       1.00      

In [7]:

joblib.dump(zone_model, os.path.join(MODEL_DIR, "zone_model.pkl"))
joblib.dump(zone_encoder, os.path.join(MODEL_DIR, "zone_label_encoder.pkl"))

joblib.dump(severity_model, os.path.join(MODEL_DIR, "severity_model.pkl"))
joblib.dump(severity_encoder, os.path.join(MODEL_DIR, "severity_label_encoder.pkl"))

joblib.dump(num_cols, os.path.join(MODEL_DIR, "numeric_features.pkl"))
joblib.dump(cat_cols, os.path.join(MODEL_DIR, "categorical_features.pkl"))
print(f"\nModels saved in: {MODEL_DIR}")


Models saved in: trained_models


In [8]:
flow_cols = ["flow1_cal", "flow2_cal", "flow3_cal"]
pressure_cols = ["pressure"]
diff_cols = ["diff12_cal", "diff23_cal"]
ratio_cols = ["ratio12_cal", "ratio23_cal"]


In [9]:
df_noisy = df.copy()

np.random.seed(42)

# Add percentage-based Gaussian noise
def add_relative_noise(series, noise_level=0.03):
    noise = np.random.normal(loc=0.0, scale=noise_level, size=len(series))
    return series * (1 + noise)

# Mild realistic noise
for col in flow_cols:
    if col in df_noisy.columns:
        df_noisy[col] = add_relative_noise(df_noisy[col], noise_level=0.04)

for col in pressure_cols:
    if col in df_noisy.columns:
        df_noisy[col] = add_relative_noise(df_noisy[col], noise_level=0.02)

for col in diff_cols:
    if col in df_noisy.columns:
        df_noisy[col] = add_relative_noise(df_noisy[col], noise_level=0.05)

for col in ratio_cols:
    if col in df_noisy.columns:
        # Only apply where values are not NaN
        mask = df_noisy[col].notna()
        df_noisy.loc[mask, col] = add_relative_noise(df_noisy.loc[mask, col], noise_level=0.04)


In [10]:
# Clip negatives because flow/pressure should not go below zero
numeric_cols = flow_cols + pressure_cols + diff_cols + ratio_cols
for col in numeric_cols:
    if col in df_noisy.columns:
        df_noisy[col] = df_noisy[col].clip(lower=0)

# Keep labels same
# zone_new and severity_new are unchanged intentionally

# Combine original + noisy
df_augmented = pd.concat([df, df_noisy], ignore_index=True)

# Shuffle dataset
df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

# Save
df_augmented.to_csv("phase2_augmented_dataset.csv", index=False)

In [11]:
print("Original shape:", df.shape)
print("Noisy shape:", df_noisy.shape)
print("Augmented shape:", df_augmented.shape)
print("Saved as phase2_augmented_dataset.csv")

Original shape: (1134, 12)
Noisy shape: (1134, 12)
Augmented shape: (2268, 12)
Saved as phase2_augmented_dataset.csv


In [17]:
print("\nAugmented dataset shape:", df_augmented.shape)


print("\nZone distribution:")
print(df_augmented["zone_new"].value_counts())

print("\nSeverity distribution:")
print(df_augmented["severity_new"].value_counts())


Augmented dataset shape: (2268, 12)

Zone distribution:
LEAK_2_3    1058
NORMAL       718
LEAK_1_2     492
Name: zone_new, dtype: int64

Severity distribution:
MEDIUM    930
LOW       718
HIGH      620
Name: severity_new, dtype: int64


## Augemented Model Training 

In [19]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

# ============================================================
# SETTINGS
# ============================================================
INPUT_FILE = "hydrosense_phase2_labeled.csv"
AUGMENTED_FILE = "phase2_augmented_dataset.csv"
MODEL_DIR = "saved_models"

os.makedirs(MODEL_DIR, exist_ok=True)
np.random.seed(42)

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv(INPUT_FILE)

print("Original dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# ============================================================
# OPTIONAL: CLEAN AGAIN FOR SAFETY
# ============================================================
df.columns = df.columns.str.strip()

# Keep only rows having target labels
df = df.dropna(subset=["zone_new", "severity_new"]).copy()

# Convert numeric columns safely
numeric_cols = [
    "flow1", "flow2", "flow3", "pressure",
    "diff12", "diff23", "ratio12", "ratio23",
    "flow1_cal", "flow2_cal", "flow3_cal",
    "diff12_cal", "diff23_cal", "ratio12_cal", "ratio23_cal"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing ratio values if needed
for col in ["ratio12", "ratio23", "ratio12_cal", "ratio23_cal"]:
    if col in df.columns:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)

# ============================================================
# CREATE AUGMENTED / NOISY DATASET
# ============================================================
df_noisy = df.copy()

flow_cols = ["flow1_cal", "flow2_cal", "flow3_cal"]
pressure_cols = ["pressure"]
diff_cols = ["diff12_cal", "diff23_cal"]
ratio_cols = ["ratio12_cal", "ratio23_cal"]

def add_relative_noise(series, noise_level=0.03):
    """Add percentage-based Gaussian noise."""
    noise = np.random.normal(loc=0.0, scale=noise_level, size=len(series))
    return series * (1 + noise)

# Mild realistic noise
for col in flow_cols:
    if col in df_noisy.columns:
        mask = df_noisy[col].notna()
        df_noisy.loc[mask, col] = add_relative_noise(df_noisy.loc[mask, col], noise_level=0.04)

for col in pressure_cols:
    if col in df_noisy.columns:
        mask = df_noisy[col].notna()
        df_noisy.loc[mask, col] = add_relative_noise(df_noisy.loc[mask, col], noise_level=0.02)

for col in diff_cols:
    if col in df_noisy.columns:
        mask = df_noisy[col].notna()
        df_noisy.loc[mask, col] = add_relative_noise(df_noisy.loc[mask, col], noise_level=0.05)

for col in ratio_cols:
    if col in df_noisy.columns:
        mask = df_noisy[col].notna()
        df_noisy.loc[mask, col] = add_relative_noise(df_noisy.loc[mask, col], noise_level=0.04)

# Clip negative values
clip_cols = flow_cols + pressure_cols + diff_cols + ratio_cols
for col in clip_cols:
    if col in df_noisy.columns:
        df_noisy[col] = df_noisy[col].clip(lower=0)

# Combine original + noisy
df_aug = pd.concat([df, df_noisy], ignore_index=True)
df_aug = df_aug.sample(frac=1, random_state=42).reset_index(drop=True)

# Save augmented dataset
df_aug.to_csv(AUGMENTED_FILE, index=False)

print("\nAugmented dataset shape:", df_aug.shape)
print("Saved augmented dataset to:", AUGMENTED_FILE)

print("\nZone distribution:")
print(df_aug["zone_new"].value_counts())

print("\nSeverity distribution:")
print(df_aug["severity_new"].value_counts())

# ============================================================
# FEATURE SELECTION
# ============================================================
# Best to use calibrated features for training
feature_cols = [
    "flow1_cal", "flow2_cal", "flow3_cal",
    "pressure",
    "diff12_cal", "diff23_cal",
    "ratio12_cal", "ratio23_cal"
]

# Keep only features that actually exist
feature_cols = [col for col in feature_cols if col in df_aug.columns]

print("\nSelected features:", feature_cols)

# Drop rows with missing feature values
train_df = df_aug.dropna(subset=feature_cols + ["zone_new", "severity_new"]).copy()

print("\nTraining dataset shape after dropping NaNs:", train_df.shape)

X = train_df[feature_cols].copy()

# ============================================================
# ENCODE LABELS
# ============================================================
zone_encoder = LabelEncoder()
severity_encoder = LabelEncoder()

y_zone = zone_encoder.fit_transform(train_df["zone_new"])
y_severity = severity_encoder.fit_transform(train_df["severity_new"])

# Save encoders and features
joblib.dump(zone_encoder, os.path.join(MODEL_DIR, "zone_label_encoder.pkl"))
joblib.dump(severity_encoder, os.path.join(MODEL_DIR, "severity_label_encoder.pkl"))
joblib.dump(feature_cols, os.path.join(MODEL_DIR, "feature_columns.pkl"))

# ============================================================
# TRAIN TEST SPLIT
# ============================================================
X_train_zone, X_test_zone, y_train_zone, y_test_zone = train_test_split(
    X, y_zone, test_size=0.2, random_state=42, stratify=y_zone
)

X_train_sev, X_test_sev, y_train_sev, y_test_sev = train_test_split(
    X, y_severity, test_size=0.2, random_state=42, stratify=y_severity
)

# ============================================================
# TRAIN MODELS
# ============================================================
zone_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=8,
    min_samples_leaf=4,
    random_state=42,
    class_weight="balanced"
)

severity_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=8,
    min_samples_leaf=4,
    random_state=42,
    class_weight="balanced"
)

zone_model.fit(X_train_zone, y_train_zone)
severity_model.fit(X_train_sev, y_train_sev)

# ============================================================
# EVALUATE ZONE MODEL
# ============================================================
zone_preds = zone_model.predict(X_test_zone)
zone_acc = accuracy_score(y_test_zone, zone_preds)

print("\n" + "=" * 60)
print("ZONE_MODEL RESULTS")
print("=" * 60)
print(f"Accuracy: {zone_acc:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test_zone,
    zone_preds,
    target_names=zone_encoder.classes_
))

print("Confusion Matrix:")
print(confusion_matrix(y_test_zone, zone_preds))

# ============================================================
# EVALUATE SEVERITY MODEL
# ============================================================
sev_preds = severity_model.predict(X_test_sev)
sev_acc = accuracy_score(y_test_sev, sev_preds)

print("\n" + "=" * 60)
print("SEVERITY_MODEL RESULTS")
print("=" * 60)
print(f"Accuracy: {sev_acc:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test_sev,
    sev_preds,
    target_names=severity_encoder.classes_
))

print("Confusion Matrix:")
print(confusion_matrix(y_test_sev, sev_preds))

# ============================================================
# FEATURE IMPORTANCE
# ============================================================
print("\n" + "=" * 60)
print("ZONE FEATURE IMPORTANCE")
print("=" * 60)
zone_importance = pd.Series(zone_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(zone_importance)

print("\n" + "=" * 60)
print("SEVERITY FEATURE IMPORTANCE")
print("=" * 60)
sev_importance = pd.Series(severity_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(sev_importance)

# ============================================================
# SAVE MODELS
# ============================================================
joblib.dump(zone_model, os.path.join(MODEL_DIR, "zone_model.pkl"))
joblib.dump(severity_model, os.path.join(MODEL_DIR, "severity_model.pkl"))

print("\nModels and encoders saved successfully in:", MODEL_DIR)

# ============================================================
# SAMPLE PREDICTION CHECK
# ============================================================
sample = X.iloc[[0]]
zone_sample_pred = zone_encoder.inverse_transform(zone_model.predict(sample))[0]
sev_sample_pred = severity_encoder.inverse_transform(severity_model.predict(sample))[0]

print("\nSample prediction check:")
print("Input row:")
print(sample)
print("Predicted zone:", zone_sample_pred)
print("Predicted severity:", sev_sample_pred)

Original dataset shape: (1135, 24)

Columns:
['timestamp', 'flow1', 'flow2', 'flow3', 'pressure', 'diff12', 'diff23', 'ratio12', 'ratio23', 'flag_old', 'zone_old', 'severity_old', 'status_old', 'extra', 'operating_state', 'flow1_cal', 'flow2_cal', 'flow3_cal', 'diff12_cal', 'diff23_cal', 'ratio12_cal', 'ratio23_cal', 'zone_new', 'severity_new']

Augmented dataset shape: (2270, 24)
Saved augmented dataset to: phase2_augmented_dataset.csv

Zone distribution:
LEAK_2_3    1058
NORMAL       720
LEAK_1_2     492
Name: zone_new, dtype: int64

Severity distribution:
MEDIUM    930
LOW       720
HIGH      620
Name: severity_new, dtype: int64

Selected features: ['flow1_cal', 'flow2_cal', 'flow3_cal', 'pressure', 'diff12_cal', 'diff23_cal', 'ratio12_cal', 'ratio23_cal']

Training dataset shape after dropping NaNs: (1750, 24)

ZONE_MODEL RESULTS
Accuracy: 0.9714

Classification Report:
              precision    recall  f1-score   support

    LEAK_1_2       0.94      0.96      0.95        68
    